# Exercise 2: Apply Pattern for Missing Values Handling (Strategy Pattern)

### Objective:
- Learn the **Strategy Pattern** — one unified interface, multiple interchangeable behaviors.
- Practice **inheritance** and **polymorphism** using the `abc` module.

### Concept:
In Data Science, missing values (`NaN`) can be handled in several ways:
1. Dropping rows with missing values (`DropMissing`)
2. Imputing numeric missing values with the column mean (`FillMean`)
3. Imputing categorical/text missing values with the mode (`FillMode`)

Using the Strategy Pattern allows us to switch between these strategies cleanly without modifying our core cleaning logic or writing tangled `if/else` statements.

## 1. Defining the Abstract Strategy Interface

We define `MissingValueStrategy` as an abstract base class with a required method `handle(df)`.
Any concrete strategy class must inherit from this class and implement `handle(df)`.

In [1]:
import pandas as pd
from abc import ABC, abstractmethod

class MissingValueStrategy(ABC):
    """
    Abstract Base Class representing a strategy for handling missing values.
    """
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applies the missing value strategy to the given DataFrame.
        Must be implemented by all concrete subclasses.
        """
        pass

## 2. Implementing Concrete Strategies

We create three concrete subclasses:
1. `DropMissing`: Drops any row containing missing values.
2. `FillMean`: Fills missing values in numeric columns with the column mean.
3. `FillMode`: Fills missing values with the most frequent value (mode) of each column.

In [2]:
class DropMissing(MissingValueStrategy):
    """Drops all rows with missing values."""
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[Strategy] Applying DropMissing: Dropping rows with NaN values...")
        cleaned_df = df.dropna().copy()
        return cleaned_df


class FillMean(MissingValueStrategy):
    """Fills missing numerical values with column mean."""
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[Strategy] Applying FillMean: Filling numeric missing values with mean...")
        cleaned_df = df.copy()
        numeric_cols = cleaned_df.select_dtypes(include=['number']).columns
        
        for col in numeric_cols:
            if cleaned_df[col].isnull().sum() > 0:
                mean_val = cleaned_df[col].mean()
                cleaned_df[col] = cleaned_df[col].fillna(mean_val)
                print(f"  - Filled '{col}' missing values with mean = {mean_val:.2f}")
                
        return cleaned_df


class FillMode(MissingValueStrategy):
    """Fills missing values with the most frequent value (mode)."""
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[Strategy] Applying FillMode: Filling missing values with column mode...")
        cleaned_df = df.copy()
        
        for col in cleaned_df.columns:
            if cleaned_df[col].isnull().sum() > 0:
                mode_val = cleaned_df[col].mode().iloc[0]
                cleaned_df[col] = cleaned_df[col].fillna(mode_val)
                print(f"  - Filled '{col}' missing values with mode = '{mode_val}'")
                
        return cleaned_df

## 3. Creating the DataCleaner Class

The `DataCleaner` class acts as the **Context** in the Strategy Pattern.
It maintains a reference to a strategy object and delegates the cleaning task to it.
We can dynamically swap the strategy using `set_strategy()`.

In [3]:
class DataCleaner:
    """
    Context class that applies a chosen MissingValueStrategy.
    """
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy
    
    def set_strategy(self, strategy: MissingValueStrategy):
        """Allows switching strategy at runtime."""
        self.strategy = strategy
        print(f"Switched strategy to: {strategy.__class__.__name__}")
    
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Executes the current missing value strategy."""
        if self.strategy is None:
            raise ValueError("No strategy has been set for DataCleaner!")
        return self.strategy.handle(df)

## 4. Testing the Strategy Pattern with Sample Data

In [4]:
# Load sample data
raw_df = pd.read_csv('sample_data.csv')
print('Original Raw Data:')
display(raw_df)

Original Raw Data:


,id,name,age,salary,city
0,1,Alice,25.0,50000.0,New York
1,2,Bob,30.0,NaN,Paris
2,3,Charlie,NaN,70000.0,London
3,4,David,40.0,90000.0,TOKYO
4,5,Alice,25.0,50000.0,New York
5,6,Eve,35.0,65000.0,paris
6,7,Frank,28.0,52000.0,london
7,8,Grace,NaN,58000.0,NaN


In [5]:
print('Missing values per column in original data:')
print(raw_df.isnull().sum())

Missing values per column in original data:
id        0
name      0
age       2
salary    1
city      1
dtype: int64


In [6]:
# Test Strategy 1: Drop Missing Values
cleaner = DataCleaner(strategy=DropMissing())
df_dropped = cleaner.clean(raw_df)
print('\nResult after DropMissing:')
display(df_dropped)

[Strategy] Applying DropMissing: Dropping rows with NaN values...

Result after DropMissing:


,id,name,age,salary,city
0,1,Alice,25.0,50000.0,New York
3,4,David,40.0,90000.0,TOKYO
4,5,Alice,25.0,50000.0,New York
5,6,Eve,35.0,65000.0,paris
6,7,Frank,28.0,52000.0,london


In [7]:
# Test Strategy 2: Fill Mean
cleaner.set_strategy(FillMean())
df_mean_filled = cleaner.clean(raw_df)
print('\nResult after FillMean:')
display(df_mean_filled)

Switched strategy to: FillMean
[Strategy] Applying FillMean: Filling numeric missing values with mean...
  - Filled 'age' missing values with mean = 30.50
  - Filled 'salary' missing values with mean = 62142.86

Result after FillMean:


,id,name,age,salary,city
0,1,Alice,25.0,50000.000000,New York
1,2,Bob,30.0,62142.857143,Paris
2,3,Charlie,30.5,70000.000000,London
3,4,David,40.0,90000.000000,TOKYO
4,5,Alice,25.0,50000.000000,New York
5,6,Eve,35.0,65000.000000,paris
6,7,Frank,28.0,52000.000000,london
7,8,Grace,30.5,58000.000000,NaN


In [8]:
# Test Strategy 3: Fill Mode
cleaner.set_strategy(FillMode())
df_mode_filled = cleaner.clean(raw_df)
print('\nResult after FillMode:')
display(df_mode_filled)

Switched strategy to: FillMode
[Strategy] Applying FillMode: Filling missing values with column mode...
  - Filled 'age' missing values with mode = '25.0'
  - Filled 'salary' missing values with mode = '50000.0'
  - Filled 'city' missing values with mode = 'New York'

Result after FillMode:


,id,name,age,salary,city
0,1,Alice,25.0,50000.0,New York
1,2,Bob,30.0,50000.0,Paris
2,3,Charlie,25.0,70000.0,London
3,4,David,40.0,90000.0,TOKYO
4,5,Alice,25.0,50000.0,New York
5,6,Eve,35.0,65000.0,paris
6,7,Frank,28.0,52000.0,london
7,8,Grace,25.0,58000.0,New York


### Summary:
- Successfully implemented the **Strategy Pattern** for missing values handling.
- `DataCleaner` is completely decoupled from the specific implementation details of how missing values are handled.
- New strategies (e.g., `FillMedian`, `Interpolate`) can be added easily without altering existing code (Open/Closed Principle).